In [ ]:
import os 
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"

In [ ]:
from time import perf_counter
from vllm import LLM, SamplingParams
from datasets import load_dataset
from collections import defaultdict
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
from huggingface_hub import notebook_login, snapshot_download
notebook_login()

In [ ]:
path = snapshot_download("allenai/OLMo-2-1124-7B")
print(path)

In [ ]:
path = snapshot_download("allenai/OLMo-2-0425-1B-Instruct")
print(path)

In [ ]:
path = snapshot_download("allenai/OLMoE-1B-7B-0924-Instruct")
print(path)

# Prefil Workload

In [ ]:
def construct_prompt(row):
    return {"prompt": f"{row['context']} {row['question']} \n\nAnswer with exactly one of the following strings: '{row['ans0']}', '{row['ans1']}', '{row['ans2']}'. \n\nAnswer:"}

In [ ]:
ds = load_dataset("heegyu/bbq", revision = "refs/convert/parquet", data_dir="Age", split="test")
prompts = ds.map(construct_prompt, batched=False)["prompt"]

In [ ]:
print(prompts[0])

In [ ]:
sampling_params = SamplingParams(
        n=1,
        temperature=1.0,
        top_p=0.95,
        top_k= -1,  
        repetition_penalty= 1.0,
        max_tokens=8,
        seed=42,
    )

In [ ]:
def generate_prefill(model):
    tokens_per_sec = defaultdict(list)

    llm = LLM(
        model=model,
        quantization="fp8_per_tensor",  
        kv_cache_dtype="fp8",    
        gpu_memory_utilization=0.50,
        enable_sleep_mode=True    
    )

    try:
        for prompt in tqdm(prompts, desc="Generating prefill completions"):
            torch.cuda.synchronize()
            start = perf_counter()

            output = llm.generate([prompt], sampling_params, use_tqdm=False)

            torch.cuda.synchronize()
            elapsed = perf_counter() - start
            
            len_prompt = len(output[0].prompt_token_ids)
            len_generated_tokens = len(output[0].outputs[0].token_ids)

            tokens_per_sec[len_prompt].append(len_generated_tokens/elapsed)
    finally:
        llm.sleep(level=2)
        torch.cuda.empty_cache()

    return {length: sum(rate)/len(rate) for length, rate in tokens_per_sec.items()}

In [ ]:
large = generate_prefill("allenai/OLMo-2-1124-7B")
print(large)

In [ ]:
small = generate_prefill("allenai/OLMo-2-0425-1B-Instruct")
print(small)

In [ ]:
moe = generate_prefill("allenai/OLMoE-1B-7B-0924-Instruct")
print(moe)

In [ ]:
large_x = sorted(large)
small_x = sorted(small)
moe_x = sorted(moe)

plt.figure(figsize=(8, 5))

plt.plot(large_x, [large[x] for x in large_x], marker="o", label="OLMo 7B")
plt.plot(small_x, [small[x] for x in small_x], marker="o", label="OLMo 1B")
plt.plot(moe_x, [moe[x] for x in moe_x], marker="o", label="OLMoE 1B-7B")

plt.xlabel("Input length (tokens)")
plt.ylabel("Output tokens / second")
plt.title("Generation Speed vs. Input Length")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

# Decode Workload

In [ ]:
ds = load_dataset("openai/gsm8k", split="train").select(range(25))

In [ ]:
sampling_params = SamplingParams(
        n=8,
        temperature=1.0,
        top_p=0.95,
        top_k= -1,  
        repetition_penalty= 1.0,
        max_tokens=128,
        seed=42,
    )

In [ ]:
def generate_decode(model, num_parallel_allowed):
    total_tokens_per_sec = 0

    llm = LLM(
            model=model,
            quantization="fp8_per_tensor",  
            kv_cache_dtype="fp8",    
            gpu_memory_utilization=0.50,
            enable_sleep_mode=True, 
            max_num_seqs=8,
            max_num_active_seqs=num_parallel_allowed
        )

    try:
        for prompt in tqdm(ds['question'], desc="Generating decode completions"):
            torch.cuda.synchronize()
            start = perf_counter()
            
            outputs = llm.generate([prompt], sampling_params, use_tqdm=False)
            
            torch.cuda.synchronize()
            elapsed = perf_counter() - start

            num_output_tokens = sum(len(completion.token_ids) for completion in outputs[0].outputs)

            tokens_per_sec = num_output_tokens/elapsed
            total_tokens_per_sec += tokens_per_sec
    finally:
        llm.sleep(level=2)
        torch.cuda.empty_cache() 
    

    return total_tokens_per_sec / len(ds['question'])

        

In [ ]:
large_decode = {1: generate_decode("allenai/OLMo-2-1124-7B", 1), 2: generate_decode("allenai/OLMo-2-1124-7B", 2), 4: generate_decode("allenai/OLMo-2-1124-7B", 4), 8: generate_decode("allenai/OLMo-2-1124-7B", 8)}
print(large_decode)

In [ ]:
small_decode = {1: generate_decode("allenai/OLMo-2-0425-1B-Instruct", 1), 2: generate_decode("allenai/OLMo-2-0425-1B-Instruct", 2), 4: generate_decode("allenai/OLMo-2-0425-1B-Instruct", 4), 8: generate_decode("allenai/OLMo-2-0425-1B-Instruct", 8)}
print(small_decode)

In [ ]:
moe_decode = {1: generate_decode("allenai/OLMoE-1B-7B-0924-Instruct", 1), 2: generate_decode("allenai/OLMoE-1B-7B-0924-Instruct", 2), 4: generate_decode("allenai/OLMoE-1B-7B-0924-Instruct", 4), 8: generate_decode("allenai/OLMoE-1B-7B-0924-Instruct", 8)}
print(moe_decode)

In [ ]:
parallel_gens = sorted(moe_decode.keys())

plt.figure(figsize=(8, 5))

plt.plot(
    parallel_gens,
    [large_decode[n] for n in parallel_gens],
    marker="o",
    label="OLMo 7B"
)

plt.plot(
    parallel_gens,
    [small_decode[n] for n in parallel_gens],
    marker="o",
    label="OLMo 1B"
)

plt.plot(
    parallel_gens,
    [moe_decode[n] for n in parallel_gens],
    marker="o",
    label="OLMoE 1B-7B"
)

plt.xlabel("Parallel Generations")
plt.ylabel("Output Tokens / Second")
plt.title("Decode Throughput vs. Parallel Generations")

plt.xscale("log", base=2)
plt.xticks(parallel_gens, parallel_gens)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()